In [2]:
from unsloth import FastVisionModel
from dotenv import load_dotenv
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import pandas as pd
import json
from tqdm.auto import tqdm
from PIL import Image, ImageEnhance
from transformers import EarlyStoppingCallback

/Users/zain/ocr-id-parsing/.venv/lib/python3.14/site-packages/trl/extras/best_of_n_sampler.py:24: FutureWarning: `BestOfNSampler` is deprecated and will be removed in TRL 0.25.
  class BestOfNSampler:


In [ ]:
load_dotenv()

In [ ]:
field_structure = {
    "side": "string 'front' or 'back'",
    "first_name": "string (arabic)",
    "last_name": "string (arabic)",
    "national_id": "string, 14 digits",
    "address": "string (arabic)",
    "address2": "string (arabic)",
    "issue_date": "string, formatted date (arabic)",
    "expiration_date": "string, formatted date (arabic)",
    "job_title": "string (arabic)",
    "gender": "string, 'male' or 'female' (arabic)",
    "religion": "string, 'muslim' or 'christian' (arabic)",
    "marital_status": "string, 'single', 'married' or 'widow' (arabic)"
}

In [ ]:
SYSTEM_PROMPT = f'''
    You are a Vision Language Model tasked with extracted field values from an Egyptian national identity
    document. You must extract the fields without making any changes to the fields and return them
    as they are in Arabic script. If there is something you cannot extract, do not attempt to infer it based
    on other information.
'''
USER_PROMPT = f'''
    You are given one side of an Egyptian National ID, either front or back.
    Extract all the fields that are on this side out of the ID and report which side it was
    following this format: {field_structure}.  The key order does not matter. Return all 
    fields that you detect as they appear and do not make any changes or updates to any of the fields. Return in json format.
'''

In [ ]:
def preprocess_image(image: Image.Image):
    grey_image = image.convert('L')
    enhancer = ImageEnhance.Contrast(grey_image)
    enhanced_image = enhancer.enhance(1.5)

    return enhanced_image

In [ ]:
def generate_conversation(data, datatype):

    image = preprocess_image(Image.open(f"{data_path}/{datatype}/images/{data['image']}"))

    all_fields = {
        "side": data['side'],
        "first_name": data['first_name'],
        "last_name": data['last_name'],
        "national_id": data['national_id'],
        "address": data['address'],
        "address2": data['address2'],
        "issue_date": data['issue_date'],
        "expiration_date": data['expiration_date'],
        "job_title": data['job_title'],
        "gender": data['gender'],
        "religion": data['religion'],
        "marital_status": data['marital_status'],
    }

    target = {
        key: value for key, value in all_fields.items()
        if not pd.isna(value)
    }

    message = json.dumps(target, ensure_ascii=False)

    conversation = [
        {
            'role': 'system',
            'content': [
                    {
                        'type': 'text',
                        'text': SYSTEM_PROMPT
                    }
            ]
        },
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text', 'text': USER_PROMPT 
                },
                {
                    'type': 'image', 
                    'image': image
                }
            ]
        },
        {
            'role': 'assistant', 
            'content': [
                {
                    'type': 'text', 
                    'text': message
                }
            ]
        }
    ]
    return {"messages": conversation}

##### Load and split data

In [ ]:
data_path = './data/synthetic-ids'
train = pd.read_csv(f"{data_path}/train/IDLabels.csv")
val = pd.read_csv(f"{data_path}/val/IDLabels.csv")

#### Apply chat transformation

In [ ]:
training_data = []
for idx, sample in tqdm(train.iterrows()):
    training_data.append(generate_conversation(sample, "train"))

In [ ]:
validation_data = []
for idx, sample in tqdm(val.iterrows()):
    validation_data.append(generate_conversation(sample, "val"))

##### Load pretrained model

In [ ]:
# model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
#                                                    load_in_4bit = True,
#                                                    use_gradient_checkpointing=True)


model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit",
                                                   load_in_4bit = True,
                                                   use_gradient_checkpointing=True)

##### Set up finetuning model

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)
FastVisionModel.for_training(model)

In [3]:
args = SFTConfig(
        # training
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        learning_rate = 2e-4, 
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        num_train_epochs = 10,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",

        # eval
        per_device_eval_batch_size = 1,
        eval_strategy='steps',
        eval_steps=50,

        # output
        output_dir = "models",
        report_to = 'wandb',
        run_name = 'ocr-id-detection',

        # logging
        logging_steps = 25,
        save_steps=50,

        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 10000,
        bf16=False,

        push_to_hub=True,
        hub_private_repo=True,
        hub_model_id='zain110506/ocr-id-parser',
        hub_strategy='checkpoint'

)

NotImplementedError: Unsloth MLX: unsupported TrainingArguments/SFTConfig kwargs: hub_private_repo, hub_strategy, run_name. Remove these kwargs or use fields implemented by MLXTrainingConfig.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_data,
    eval_dataset=validation_data,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    args=args
)


#### Train the model

In [ ]:
trainer_stats = trainer.train()

#### Save the model and tokenizer

In [ ]:
model.save_pretrained("qwen3_vlm")
tokenizer.save_pretrained("qwen3_vlm")